# California Housing Prices
## Data Source
This dataset is used in [Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, 3rd Edition](https://learning.oreilly.com/library/view/hands-on-machine-learning/9781098125967/) by Aurélien Géron.  
The description of the dataset can be found [here](https://github.com/ageron/data/tree/main/housing)
## Goal
Apply sevearl feature engineering measures to the raw dataset and build a ML model that takes in the engineered featrues to predict house prices
## Structue
1. Load Data
2. Perform Preliminary EDA
3. Create the Test Set
4. Perform In-depth EDA on Train Set
5. Prepare Data  

**Note**: _The notebook basically follows [Mr. Géron's notebook for the book's demostration](https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb), with some additional steps that I deemeed helpful for exploring ScikitLearn and improving the model's performance._


In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
import shutil

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load data
## Download data
**Note**: You mind need to close VPN if you have one

In [ ]:


def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        with urllib.request.urlopen(url) as response:
            shutil.copyfileobj(response, tarball_path.open("wb"))
    with tarfile.open(tarball_path) as housing_tarball:
        housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

## 2. Perform Preliminary EDA
This step is pure metadata inspection (e.g., column names, data types, obvious impossible values, missing value patterns)

In [ ]:
housing.info()

1. `total_bedrooms` has 207 missing values

### Numeric Columns

In [ ]:
housing.describe()

1. According to [the Wikepdia page for California](https://en.wikipedia.org/wiki/California), the range of longitude and latitude of California is:
     - Latitude: 32°32′ N to 42° N
     - Longitude: 114°8′ W to 124°26′ W  
    -> Somehow the min value of longtitude is to the west of the west bound, but given it's not a severe off, we let it be.
2. 


In [ ]:
housing.hist(bins=50, figsize=(20,15))
plt.show()

1. The histgrams of `total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`, and `median_house_value` all show a right-skewed distrubtion. (`median_house_value` has the values capped at 500001, otherwise we can also observe such skew). 
2. Usually non-negative variables that are of aggregated counts behave this way.
3. For these distrubtions, it's typicallly appropriate to log-transform them before feeding them to the model. Because doing so we can reduce skewness through compressing extreme values and thus make the relationships more linear/stable.

### Non-numeric Columns

In [ ]:
housing["ocean_proximity"].value_counts()

It seems only a few blocks are categorized as `ISLAND`. This might suggest that model would struggle to learnt the pattern for this category. We can perform some transformation to merge this categories with others if deemed necessary.

# 3. Create the Test Set

To avoid in balance of the label between the train and test sets, we bin the median income into income category and stratefy based on it (but we don't use income category as a feature)

In [ ]:
from sklearn.model_selection import train_test_split

housing["income_cat"] = pd.cut(housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)


train_set, test_set = train_test_split(
    housing, 
    test_size=0.2, 
    stratify=housing["income_cat"],
    random_state=42
)

# drop income_cat after splitting
for set in (train_set, test_set):
    set.drop("income_cat", axis=1, inplace=True)

# have a quick view on the split
train_set.shape, test_set.shape

In [ ]:
# replace housing with the train set
housing = train_set

# 4. Perform In-depth EDA on the Train Set

## Visualize Geo data

In [ ]:
# Download the California image
IMAGES_PATH = Path() / "images" / "end_to_end_project"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)
filename = "california.png"
if not (IMAGES_PATH / filename).is_file():
    homl3_root = "https://github.com/ageron/handson-ml3/raw/main/"
    url = homl3_root + "images/end_to_end_project/" + filename
    print("Downloading", filename)
    urllib.request.urlretrieve(url, IMAGES_PATH / filename)

In [ ]:
# plot the housing data on the California map with population as the size of the circle and median house value as the color
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
    s=housing["population"] / 100, label="population", figsize=(10, 7),
    c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
)

california_img = plt.imread(IMAGES_PATH / filename)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)

plt.legend()
plt.show()


1. We can clearly see that the major cities (San Francisco, Santa Babara, Los Angeles, and San Diego) have clustered blocks, higher median house values, and higher population.
2. On the other hand, the Central Valley area has quite concentrated population but relatively lower median house value (apart from Sacramento)
3. Lake Tahoe has a small cluster of mid housing price — becuase it's a premier year-round resort destination (winter skiing and summer water activities)
4. Generally speaking, the housing price is higher when getting closer the Pacific ocean

In [ ]:
# check ocean proximity on the map

sns.scatterplot(data=housing, x="longitude", y="latitude", hue="ocean_proximity", 
alpha=0.4)
california_img = plt.imread(IMAGES_PATH / filename)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)

plt.legend()
plt.show()

1. Only the San Francisco Bay Area has the `NEAR BAY` category
2. The `ISLAND` category only appears on Santa Catalina but not other Channel Islands (could be a result of aggregation)

In [ ]:
# see how median income is distributed on the map
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=1,
    s=housing["population"] / 100, label="population", figsize=(10, 7),
    c=housing["median_income"] * 10000, cmap=plt.get_cmap("jet"), colorbar=True,
)

california_img = plt.imread(IMAGES_PATH / filename)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)

plt.legend()
plt.show()

It seems the geo distribution of the median income is similar to the one of the housing price

## Visualize correlations

In [ ]:
# see the correlation between the features and the target
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)


1. The `median_income` is the most correlated feature with the median house value
2. The `total_rooms`, `housing_median_age`, and `households` are also positively correlated with the median house value, but the correlation is wewak
3. The `total_bedrooms`, `population`, and `longitude` are negatively correlated with the median house value, but, again, the correlation is weak

In [ ]:
# visualize the correlation matrix
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

1. `total_rooms`, `total_bedrooms`, `populaiton`, and `households` have strong postivie linear correlation with each other, which would usually lead to multicollinearity issue if we use a unregularized linear model. Why? because having multiple features that are can be predicted by one another would introduce instability in coefficients — small data changes can cause major swings in coefficient values. So, we usually need to introducing regularization (i.e. forcing the model to generalize by adding penalty terms to the loss function)
    - Below I will do a experiment on various pipelines to show-case this
2. `longitude` and `latitude` has a strong negative correlation because California basically goes from NW to SE
3. `total_rooms`, `total_bedrooms`, `populaiton`, and `households` have a medium negative correlation with `housing_median_age`

# 5. Prepare Data

## Imputation

In [ ]:
# impute missing values in `total_bedrooms`
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing_num = housing.drop("ocean_proximity", axis=1)
imputer.fit(housing_num)
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)

In [ ]:
# see the imputed values
for feature, stats in zip(imputer.feature_names_in_, imputer.statistics_):
    print(f"{feature}: {stats}")

In [ ]:
# see the strategy used to impute missing values
imputer.strategy

# 6. ???

## Setup and imports